# 01 · Limpieza y auditoría de la base IRCA (SIVICAP)

**Proyecto:** Calidad del agua para consumo humano en el Cauca, 2007–2025.
**Fuente:** SIVICAP – Ministerio de Salud y Protección Social / INS, conjunto `nxt2-39c3` de datos.gov.co.
**Consulta:** 25/11/2026.

Este notebook documenta la descarga, la normalización, la auditoría y la exportación de las tablas limpias. La lógica vive en `src/cleaning.py` para que el proceso sea reproducible desde la terminal con `python -m src.cleaning`.

## 1. Preparación del entorno

El notebook se puede ejecutar desde la raíz del proyecto o desde `notebooks/`. El bloque siguiente ajusta el directorio de trabajo y el `sys.path`.

In [1]:
import os
import sys
from pathlib import Path

BASE = Path.cwd()
if BASE.name == "notebooks":
    BASE = BASE.parent
    os.chdir(BASE)
sys.path.insert(0, str(BASE))

import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 170)
print("Directorio de trabajo:", Path.cwd())

Directorio de trabajo: C:\Users\chris\Desktop\electiva-profesional-ii-trabajo-final


## 2. Carga de la base cruda

La descarga original se hizo con la API de datos.gov.co:

```bash
curl -L "https://www.datos.gov.co/resource/nxt2-39c3.json?$limit=50000" \
  -o data/raw/irca_colombia_raw.json
```

Se usa JSON porque el endpoint CSV de exportación estaba deprecado al momento de la consulta. El archivo tiene 20.212 registros.

In [2]:
from src.cleaning import cargar_crudo

crudo = cargar_crudo()
print("Filas:", crudo.shape[0], "| Columnas:", crudo.shape[1])
crudo.head(3)

Filas: 20212 | Columnas: 11


,departamentocodigo,departamento,municipiocodigo,municipio,a_o,irca,nivel_de_riesgo,ircaurbano,nivel_de_riesgo_urbano,ircarural,nivel_de_riesgo_rural
0,11,"Bogotá, D.C.",#TODOS,#TODOS,2024,3.6,Sin riesgo,1.1,Sin riesgo,14.8,Riesgo medio
1,13,Bolívar,#TODOS,#TODOS,2024,11.3,Bajo riesgo,12.4,Bajo riesgo,6.5,Bajo riesgo
2,15,Boyacá,#TODOS,#TODOS,2024,7.5,Bajo riesgo,4.8,Sin riesgo,22,Riesgo medio


### 2.1. Columnas, tipos y nulos de la base original

In [3]:
resumen_crudo = pd.DataFrame(
    {
        "tipo": crudo.dtypes.astype(str),
        "nulos": crudo.isna().sum(),
        "ejemplo": [
            crudo[c].dropna().iloc[0] if crudo[c].notna().any() else None
            for c in crudo.columns
        ],
    }
)
resumen_crudo

,tipo,nulos,ejemplo
departamentocodigo,str,0,11
departamento,str,0,"Bogotá, D.C."
municipiocodigo,str,0,#TODOS
municipio,str,0,#TODOS
a_o,str,0,2024
irca,str,0,3.6
nivel_de_riesgo,str,0,Sin riesgo
ircaurbano,str,0,1.1
nivel_de_riesgo_urbano,str,0,Sin riesgo
ircarural,str,0,14.8


## 3. Limpieza

Decisiones principales:

- `ND`, `N/D`, `Sin dato` y cadenas vacías se convierten en nulos. **No se imputan** valores rurales.
- Las variantes `Bajo riesgo` y `Alto riesgo` se unifican con las categorías oficiales.
- Los códigos DANE se conservan como texto para no perder ceros a la izquierda.
- La fila `#TODOS` se separa de la base municipal para evitar doble conteo.
- Se valida la categoría reportada contra la calculada con los umbrales oficiales (5, 14, 35 y 80).

In [4]:
from src.cleaning import limpiar

limpio = limpiar(crudo)
print("Filas:", limpio.shape[0], "| Columnas:", limpio.shape[1])
limpio.head(3)

Filas: 20212 | Columnas: 16


,codigo_departamento,departamento,codigo_municipio,municipio,anio,irca,nivel_riesgo,irca_urbano,nivel_riesgo_urbano,irca_rural,nivel_riesgo_rural,categoria_calculada,coincide_categoria,tiene_dato_rural,brecha_rural_urbano,es_agregado
0,91,Amazonas,#TODOS,#TODOS,2007,25.5,Riesgo medio,25.5,Riesgo medio,<NA>,NaN,Riesgo medio,True,False,<NA>,True
1,91,Amazonas,#TODOS,#TODOS,2008,25.4,Riesgo medio,25.6,Riesgo medio,0.0,Sin riesgo,Riesgo medio,True,True,-25.6,True
2,91,Amazonas,#TODOS,#TODOS,2009,24.2,Riesgo medio,24.2,Riesgo medio,<NA>,NaN,Riesgo medio,True,False,<NA>,True


## 4. Auditoría de calidad

In [5]:
from src.cleaning import auditar

auditoria = auditar(limpio, crudo)
auditoria

,chequeo,resultado,detalle,tratamiento
0,Registros totales,20212,Base nacional descargada de datos.gov.co,Se conserva la base completa como referencia
1,Registros del Cauca,736,"Municipios del Cauca, sin la fila agregada #TODOS",Base de análisis municipal
2,Filas agregadas del Cauca (#TODOS),19,Una fila por año con el agregado departamental,Se separan para no duplicar el conteo municipal
3,Municipios del Cauca,42,Municipios distintos en la base municipal,Se validan contra el catálogo DANE de 42 munic...
4,Periodo,2007-2025,Años cubiertos por la base,Se usa 2007-2025 para la serie y 2025 para el ...
5,Duplicados exactos (toda la base),0,Filas idénticas en todas las columnas,No se eliminan si no hay evidencia de error; s...
6,Duplicados municipio-año en el Cauca,0,La llave municipio-año debería ser única,Se verificó que no hay duplicados
7,IRCA total nulo en el Cauca,0,Valores ND o vacíos después de convertir a número,Se excluyen del cálculo del indicador
8,IRCA urbano nulo en el Cauca,5,Valores ND o vacíos,Se mantiene como nulo; no se imputa
9,IRCA rural nulo en el Cauca,489,66.4% de los registros municipales,No se imputa; el análisis rural se limita a ca...


## 5. Base municipal del Cauca

La base municipal excluye la fila agregada `#TODOS`. La serie departamental se conserva aparte.

In [6]:
cauca = limpio[(limpio["departamento"] == "Cauca") & (~limpio["es_agregado"])].copy()
cauca_dep = limpio[(limpio["departamento"] == "Cauca") & (limpio["es_agregado"])].copy()

print("Cauca municipal:", cauca.shape)
print("Municipios:", cauca["municipio"].nunique())
print("Años:", int(cauca["anio"].min()), "-", int(cauca["anio"].max()))
print("Cauca #TODOS:", cauca_dep.shape)
cauca.head(3)

Cauca municipal: (736, 16)
Municipios: 42
Años: 2007 - 2025
Cauca #TODOS: (19, 16)


,codigo_departamento,departamento,codigo_municipio,municipio,anio,irca,nivel_riesgo,irca_urbano,nivel_riesgo_urbano,irca_rural,nivel_riesgo_rural,categoria_calculada,coincide_categoria,tiene_dato_rural,brecha_rural_urbano,es_agregado
7502,19,Cauca,19022,Almaguer,2007,62.9,Riesgo alto,62.9,Riesgo alto,<NA>,NaN,Riesgo alto,True,False,<NA>,False
7503,19,Cauca,19022,Almaguer,2008,71.1,Riesgo alto,71.5,Riesgo alto,69.2,Riesgo alto,Riesgo alto,True,True,-2.3,False
7504,19,Cauca,19022,Almaguer,2009,71.7,Riesgo alto,71.7,Riesgo alto,<NA>,NaN,Riesgo alto,True,False,<NA>,False


### 5.1. Completitud de las variables principales

In [7]:
faltantes = pd.DataFrame(
    {
        "nulos": cauca[["irca", "irca_urbano", "irca_rural"]].isna().sum(),
        "porcentaje": (
            cauca[["irca", "irca_urbano", "irca_rural"]].isna().mean() * 100
        ).round(1),
    }
)
faltantes

,nulos,porcentaje
irca,0,0.0
irca_urbano,5,0.7
irca_rural,489,66.4


La variable rural es la más incompleta: 489 de 736 registros (66,4%) no tienen IRCA rural. Por eso el análisis de brecha se limita a los casos completos y se declara el sesgo.

### 5.2. Cobertura municipal por año

In [8]:
cobertura = (
    cauca.groupby("anio")
    .agg(municipios=("municipio", "nunique"), con_rural=("irca_rural", "count"))
    .reset_index()
)
cobertura

,anio,municipios,con_rural
0,2007,32,5
1,2008,42,9
2,2009,42,13
3,2010,42,24
4,2011,41,18
5,2012,41,20
6,2013,41,17
7,2014,40,16
8,2015,40,13
9,2016,40,6


### 5.3. Validación de categorías

In [9]:
print("Registros con categoría inconsistente:", int((~limpio["coincide_categoria"]).sum()))
print("\nDistribución de categorías en el Cauca:")
print(cauca["nivel_riesgo"].value_counts(dropna=False).to_string())

Registros con categoría inconsistente: 0

Distribución de categorías en el Cauca:
nivel_riesgo
Sin riesgo                 251
Riesgo bajo                239
Riesgo medio               116
Riesgo alto                 87
Inviable sanitariamente     43


## 6. Exportación

Se ejecuta el flujo oficial para guardar todas las tablas limpias, el diccionario y la auditoría. Los archivos quedan en `data/clean/` y `reports/tablas/`.

In [10]:
from src.cleaning import main

main()

Cargando base cruda...


  Registros: 20,212
Limpiando...


Auditando calidad...
Separando tablas...
Guardando archivos...



Resumen:
  Base limpia completa: 20,212 registros
  Cauca municipal:      736 registros
  Cauca departamento:   19 registros
  Nacional municipal 2025: 1,019 registros
  Nacional departamental 2025: 33 registros
  Municipios del Cauca: 42
  Años: 2007-2025

Archivos generados en data/clean y reports/tablas.


## 7. Archivos generados

| Archivo | Contenido |
|---|---|
| `data/clean/irca_colombia_clean.csv` | Base nacional limpia (20.212 registros) |
| `data/clean/irca_cauca_municipal.csv` | Base municipal del Cauca (736 registros, 42 municipios) |
| `data/clean/irca_cauca_departamento.csv` | Serie agregada `#TODOS` del Cauca (19 años) |
| `data/clean/irca_nacional_municipal_2025.csv` | Contexto nacional municipal 2025 (1.019 municipios) |
| `data/clean/irca_nacional_departamental_2025.csv` | Contexto nacional departamental 2025 (33 departamentos) |
| `data/clean/diccionario_datos.csv` | Diccionario de variables |
| `reports/tablas/calidad_datos.csv` | Auditoría de calidad |

**Limitación central:** Popayán solo tiene datos 2008–2019; en 2025 faltan Balboa, Florencia y López de Micay. Los promedios y rankings no representan a los 42 municipios en todos los años.